In [2]:
import torch 
import numpy as np

from torch import nn
from auditory_prf.prf_pipeline.torch_pipeline import CFSelector

In [ ]:
class CFInterpolateSelector(nn.Module):
    _A: float = 165.4   # Greenwood A (Hz)
    _k: float = 0.88    # Greenwood k
    _a: float = 2.1     # Greenwood a (position units)

    def __init__(self, cf_hz_array, init_cf_hz, hz_to_scalepos, scalepos_to_hz):
        # to_scalepos and from_scalepos means the position in scale, whichever we 're using. It's cochlear position in the case of Greenwood, it's ERB-rate unit in that case. 
        super().__init__()
        cf_hz_array = np.asarray(cf_hz_array, dtype=np.float64)
        n_cf = len(cf_hz_array)

        # Convert to the desired scale's position-space. It must be evenly constructed (which ERB and Greenwood does differently)
        scalepos_map = hz_to_scalepos(cf_hz_array)
        self.register_buffer('scalepos_map', torch.from_numpy(scalepos_map.astype(np.float32)))
        self.n_cf = n_cf
        self.scalepos_min = float(scalepos_map[0])
        self.scalepos_max = float(scalepos_map[-1])
        self.scalepos_step = (self.scalepos_max - self.scalepos_min) / (n_cf - 1)

        # Parameterizing in [0,1] via logit so the optimizer is unconstrained and we don't have a problem of gradients being 0 at the edges 
        init_hz = float(np.clip(init_cf_hz, float(cf_hz_array[0]), float(cf_hz_array[-1]))) # So if a CF value outside of range is entered, it gets pushed to the boundary values. 
        init_scalepos = hz_to_scalepos(init_hz)
        init_t = float(np.clip((init_scalepos - self.scalepos_min) / (self.scalepos_max - self.scalepos_min), 1e-6, 1 - 1e-6))